In [1]:
!pip install -q torch==2.1.0 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate bitsandbytes trl evaluate sacrebleu rouge-score

ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.5 MB/s eta 0:00:00


In [2]:
!pip install -q codebleu tree-sitter-java

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.2/546.2 kB 15.1 MB/s eta 0:00:00


In [3]:
import os, csv, time, warnings
import numpy as np
import torch
import torch.nn.functional as F
warnings.filterwarnings('ignore')

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
import bitsandbytes as bnb

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} '
          f'({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)')

PyTorch : 2.10.0+cu128
CUDA    : True
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)


In [4]:
# ── Config ─────────────────────────────────────────────────────────────────────
STUDENT_ID    = 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'
OUTPUT_DIR    = './distilled-tinyllama'
LOG_FILE      = './distill_log.csv'

# !! Point this at your Kaggle Dataset mount path !!
LOGITS_DIR = '/kaggle/input/datasets/habibajahanpriya/teacher-logits/teacher_logits'

# Same offsets used in Notebook 1 — do not change
SPLIT_OFFSET  = {'train': 0, 'validation': 1_000_000, 'test': 2_000_000}

DEVICE        = 'cuda:0'
MAX_SEQ_LEN   = 128
EPOCHS        = 3
LR            = 5e-5
# TRAIN_SAMPLES = 10_000
# TRAIN_SAMPLES = 10
BATCH_SIZE    = 2
GRAD_ACCUM    = 8           # effective batch = 16
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01
MAX_GRAD_NORM = 1.0
# EVAL_STEPS    = 500
EVAL_STEPS    = 2000

# Distillation hyper-params
TEMPERATURE   = 4.0   # softens teacher distribution; typical range 2–6
ALPHA         = 0   # weight on KL loss; (1-ALPHA) on CE loss

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(42)
np.random.seed(42)

print(f'Temperature : {TEMPERATURE}')
print(f'Alpha (KL)  : {ALPHA}  |  1-Alpha (CE): {1 - ALPHA}')
print(f'Logits dir  : {LOGITS_DIR}')

Temperature : 4.0
Alpha (KL)  : 0  |  1-Alpha (CE): 1
Logits dir  : /kaggle/input/datasets/habibajahanpriya/teacher-logits/teacher_logits


In [5]:
PROMPT_TEMPLATE = (
    'Fix the following buggy Java code:\n'
    '### Buggy Code:\n{buggy}\n'
    '### Fixed Code:\n{fixed}'
)
INFERENCE_TEMPLATE = (
    'Fix the following buggy Java code:\n'
    '### Buggy Code:\n{buggy}\n'
    '### Fixed Code:\n'
)

In [6]:
print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print(f'Vocab size: {tokenizer.vocab_size}')

Loading tokenizer ...


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Vocab size: 32000


In [7]:
print('Loading TinyLlama student in fp16 ...')
student = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map={'': DEVICE},
    trust_remote_code=True,
)
student.gradient_checkpointing_enable()
student.config.use_cache = False
student.config.pretraining_tp = 1

trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Trainable params : {trainable/1e9:.2f}B')
print(f'Student device   : {next(student.parameters()).device}')

Loading TinyLlama student in fp16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Trainable params : 1.10B
Student device   : cuda:0


In [8]:
print('Loading dataset ...')
dataset = load_dataset('code_x_glue_cc_code_refinement', 'small')

# if TRAIN_SAMPLES and TRAIN_SAMPLES < len(dataset['train']):
#     dataset['train'] = dataset['train'].select(range(TRAIN_SAMPLES))
# if TRAIN_SAMPLES and TRAIN_SAMPLES < len(dataset['validation']):
#     dataset['validation'] = dataset['validation'].select(range(TRAIN_SAMPLES))
def tokenize_fn(examples):
    texts = [
        PROMPT_TEMPLATE.format(buggy=b.strip(), fixed=f.strip())
        for b, f in zip(examples['buggy'], examples['fixed'])
    ]
    enc = tokenizer(
        texts, max_length=MAX_SEQ_LEN, truncation=True,
        padding='max_length', return_tensors=None,
    )
    enc['labels'] = enc['input_ids'].copy()
    return enc

print('Tokenising ...')
train_ds = dataset['train'].map(
    tokenize_fn, batched=True, remove_columns=dataset['train'].column_names
)
val_ds = dataset['validation'].map(
    tokenize_fn, batched=True, remove_columns=dataset['validation'].column_names
)
train_ds.set_format(type='torch')
val_ds.set_format(type='torch')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

Loading dataset ...


README.md: 0.00B [00:00, ?B/s]

small/train-00000-of-00001.parquet:   0%|          | 0.00/4.72M [00:00<?, ?B/s]

small/validation-00000-of-00001.parquet:   0%|          | 0.00/591k [00:00<?, ?B/s]

small/test-00000-of-00001.parquet:   0%|          | 0.00/589k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/5835 [00:00<?, ? examples/s]

Tokenising ...


Map:   0%|          | 0/46680 [00:00<?, ? examples/s]

Map:   0%|          | 0/5835 [00:00<?, ? examples/s]

Train: 46680 | Val: 5835


In [9]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 23340 | Val batches: 2918


In [10]:
optimizer = bnb.optim.AdamW8bit(
    student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-6
)
total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Optimizer    : 8-bit AdamW')
print(f'Total steps  : {total_steps}')
print(f'Warmup steps : {warmup_steps}')

Optimizer    : 8-bit AdamW
Total steps  : 8751
Warmup steps : 875


In [11]:
# ── Distillation loss ──────────────────────────────────────────────────────────
def distillation_loss(student_logits, teacher_logits, labels,
                      temperature=TEMPERATURE, alpha=ALPHA):
    B, T, V = student_logits.shape        # V = 32000 (student vocab)

    # teacher_logits is already truncated to (B, T, 32000) by topk_to_full
    s_log  = F.log_softmax((student_logits / temperature).float().view(-1, V), dim=-1)
    t_prob = F.softmax((teacher_logits / temperature).float().view(-1, V), dim=-1)
    kl = F.kl_div(s_log, t_prob, reduction='batchmean', log_target=False) * (temperature ** 2)

    shift_logits = student_logits[:, :-1, :].contiguous().view(-1, V)
    shift_labels = labels[:, 1:].contiguous().view(-1)
    ce = F.cross_entropy(shift_logits.float(), shift_labels, ignore_index=-100)

    return alpha * kl + (1.0 - alpha) * ce, kl.item(), ce.item()

print('distillation_loss() defined.')

distillation_loss() defined.


In [12]:
# ── Load pre-saved teacher logits from disk ────────────────────────────────────
def topk_to_full(payload, fill=-1e4, student_vocab=32000):
    T, K = payload['values'].shape
    V    = payload['vocab_size']          # 32016 (teacher)
    
    # Build full teacher vocab tensor first
    out = torch.full((T, V), fill, dtype=torch.float32)
    out.scatter_(1, payload['indices'].long(), payload['values'].float())
    
    # Truncate to student vocab size (drop the extra 16 tokens)
    return out[:, :student_vocab]   


def load_teacher_logits(split_name: str, local_indices: torch.Tensor) -> torch.Tensor:
    """
    Load a batch of teacher logits from disk.
    local_indices : 0-based positions within the split (from DataLoader batch index).
    Returns (B, T, V) fp16 tensor on DEVICE.
    """
    offset  = SPLIT_OFFSET[split_name]
    tensors = []
    for idx in local_indices:
        global_idx = offset + idx.item()
        payload = torch.load(
            os.path.join(LOGITS_DIR, f'logit_{global_idx}.pt'),
            map_location='cpu',
        )
        if isinstance(payload, dict) and 'values' in payload:
            tensors.append(topk_to_full(payload))
        else:
            tensors.append(payload.float())
    return torch.stack(tensors).half().to(DEVICE)

print('load_teacher_logits() defined.')

load_teacher_logits() defined.


In [13]:
# Add row index to each split so DataLoader batches carry it
train_ds = train_ds.add_column('__index__', list(range(len(train_ds))))
val_ds   = val_ds.add_column('__index__', list(range(len(val_ds))))
train_ds.set_format(type='torch')
val_ds.set_format(type='torch')

# Re-create loaders with the new column present
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
print('Index column added. Loaders recreated.')

Index column added. Loaders recreated.


In [14]:
def evaluate(loader, split_name='validation'):
    student.eval()
    totals, kls, ces = [], [], []
    with torch.no_grad():
        for vbatch in loader:
            vids   = vbatch['input_ids'].to(DEVICE)
            vmask  = vbatch['attention_mask'].to(DEVICE)
            vlabs  = vbatch['labels'].to(DEVICE)
            vidxs  = vbatch['__index__']

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                s_out = student(input_ids=vids, attention_mask=vmask)

            t_logits = load_teacher_logits(split_name, vidxs)
            loss, kl, ce = distillation_loss(s_out.logits, t_logits, vlabs)
            totals.append(loss.item()); kls.append(kl); ces.append(ce)

    return {'total': float(np.mean(totals)),
            'kl':    float(np.mean(kls)),
            'ce':    float(np.mean(ces))}

print('evaluate() defined.')

evaluate() defined.


In [15]:
# ── Training loop ──────────────────────────────────────────────────────────────
log_fields = ['epoch','step','train_loss','kl_loss','ce_loss',
              'val_loss','val_kl','val_ce','lr','time_s']
log_file   = open(LOG_FILE, 'w', newline='')
log_writer = csv.DictWriter(log_file, fieldnames=log_fields)
log_writer.writeheader()

best_val_loss = float('inf')
global_step   = 0

print('Starting distillation ...\n')
print(f"{'Epoch':>5} {'Step':>6} {'Train':>9} {'KL':>8} {'CE':>8} {'Val':>9} {'LR':>10}")
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    student.train()
    running = {'total': 0.0, 'kl': 0.0, 'ce': 0.0}
    t0 = time.time()
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader, 1):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        indices        = batch['__index__']

        # Load teacher logits from disk (no teacher model needed!)
        teacher_logits = load_teacher_logits('train', indices)

        # Student forward + distillation loss
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            s_out = student(input_ids=input_ids, attention_mask=attention_mask)
            loss, kl, ce = distillation_loss(s_out.logits, teacher_logits, labels)
            loss = loss / GRAD_ACCUM

        loss.backward()
        running['total'] += loss.item() * GRAD_ACCUM
        running['kl']    += kl
        running['ce']    += ce

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % EVAL_STEPS == 0 or global_step == total_steps:
                val_m    = evaluate(val_loader, 'validation')
                avg_tot  = running['total'] / step
                avg_kl   = running['kl']    / step
                avg_ce   = running['ce']    / step
                cur_lr   = scheduler.get_last_lr()[0]
                elapsed  = time.time() - t0

                print(f"{epoch:>5} {global_step:>6} {avg_tot:>9.4f} "
                      f"{avg_kl:>8.4f} {avg_ce:>8.4f} "
                      f"{val_m['total']:>9.4f} {cur_lr:>10.2e}")

                log_writer.writerow({
                    'epoch': epoch, 'step': global_step,
                    'train_loss': round(avg_tot, 5),
                    'kl_loss':   round(avg_kl,  5),
                    'ce_loss':   round(avg_ce,  5),
                    'val_loss':  round(val_m['total'], 5),
                    'val_kl':    round(val_m['kl'],    5),
                    'val_ce':    round(val_m['ce'],    5),
                    'lr':        round(cur_lr, 9),
                    'time_s':    round(elapsed, 1),
                })
                log_file.flush()

                if val_m['total'] < best_val_loss:
                    best_val_loss = val_m['total']
                    save_path = os.path.join(OUTPUT_DIR, 'best_model')
                    student.save_pretrained(save_path)
                    tokenizer.save_pretrained(save_path)
                    print(f'  ✅ Best model saved (val={best_val_loss:.4f}) → {save_path}')

                student.train()

    ckpt = os.path.join(OUTPUT_DIR, f'checkpoint_epoch_{epoch}')
    student.save_pretrained(ckpt)
    tokenizer.save_pretrained(ckpt)
    print(f'\n💾 Epoch {epoch} checkpoint → {ckpt}\n')

log_file.close()
print(f'\nDistillation complete! Best val loss: {best_val_loss:.4f}')

Starting distillation ...

Epoch   Step     Train       KL       CE       Val         LR
------------------------------------------------------------
    1   2000    0.3116  69.8452   0.3116    0.2638   4.29e-05


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ Best model saved (val=0.2638) → ./distilled-tinyllama/best_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Epoch 1 checkpoint → ./distilled-tinyllama/checkpoint_epoch_1

    2   4000    0.2413  69.5496   0.2413    0.2490   3.02e-05


...

  ✅ Best model saved (val=0.2389) → ./distilled-tinyllama/best_model
    3   8751    0.2041  70.1152   0.2041    0.2387   0.00e+00


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ Best model saved (val=0.2387) → ./distilled-tinyllama/best_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Epoch 3 checkpoint → ./distilled-tinyllama/checkpoint_epoch_3


Distillation complete! Best val loss: 0.2387


In [16]:
# ── Inference helper ───────────────────────────────────────────────────────────


import re

def clean_prediction(decoded: str) -> str:
    """Cut generation off at the first sign of trailing hallucinated content."""
    # Cut at the first '###' marker (start of any new fabricated section)
    if '###' in decoded:
        decoded = decoded.split('###')[0]
    # Also cut at double-newline, since real single-line Java fixes rarely contain one
    decoded = decoded.split('\n\n')[0]
    return decoded.strip()


def generate_fix(buggy_code: str, max_new_tokens: int = 128) -> str:
    student.eval()
    prompt = INFERENCE_TEMPLATE.format(buggy=buggy_code.strip())
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN
    ).to(DEVICE)
    with torch.no_grad():
        outputs = student.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if '### Fixed Code:' in decoded:
        decoded = decoded.split('### Fixed Code:')[-1].strip()
    decoded = clean_prediction(decoded)
    return decoded

# Quick sanity check
sample = dataset['test'][0]
print('Buggy :', sample['buggy'][:120])
print('Pred  :', generate_fix(sample['buggy'])[:120])
print('Gold  :', sample['fixed'][:120])

Buggy : private TYPE_1 getType ( TYPE_2 VAR_1 ) { TYPE_3 VAR_2 = new TYPE_3 ( STRING_1 ) ; return new TYPE_1 ( VAR_2 , VAR_2 ) ;
Pred  : private TYPE_1 getType ( TYPE_2 VAR_1 ) { TYPE_3 VAR_2 = new TYPE_3 ( STRING_1 ) ; return new TYPE_1 ( VAR_2 , VAR_2 , V
Gold  : private TYPE_1 getType ( TYPE_2 VAR_1 ) { TYPE_3 VAR_2 = new TYPE_3 ( STRING_1 ) ; return new TYPE_1 ( VAR_2 , VAR_2 , t


In [17]:
import evaluate as hf_evaluate
from rouge_score import rouge_scorer as rouge_scorer_lib

MAX_TEST_SAMPLES = 5800
test_subset = dataset['test'].select(range( MAX_TEST_SAMPLES))

predictions, references = [], []
student.eval()
for ex in tqdm(test_subset, desc='Inference'):
    predictions.append(generate_fix(ex['buggy']))
    references.append(ex['fixed'].strip())

sacrebleu = hf_evaluate.load('sacrebleu')
rouge_sc   = rouge_scorer_lib.RougeScorer(['rougeL'], use_stemmer=False)

bleu_result  = sacrebleu.compute(predictions=predictions, references=[[r] for r in references])
rouge_scores = [rouge_sc.score(ref, pred)['rougeL'].fmeasure
                for pred, ref in zip(predictions, references)]
rouge_l = np.mean(rouge_scores) * 100

exact_matches = [pred.strip() == ref.strip() for pred, ref in zip(predictions, references)]
exact_match = np.mean(exact_matches) * 100

print('\n' + '─' * 50)
print('Evaluation Results')
print('─' * 50)
print(f'  SacreBLEU  : {bleu_result["score"]:.4f}')
print(f'  ROUGE-L    : {rouge_l:.4f}')
print(f'  Exact Match: {exact_match:.4f}')
print('─' * 50)

Inference:   0%|          | 0/5800 [00:00<?, ?it/s]


──────────────────────────────────────────────────
Evaluation Results
──────────────────────────────────────────────────
  SacreBLEU  : 80.2861
  ROUGE-L    : 84.1862
  Exact Match: 15.4655
──────────────────────────────────────────────────


In [18]:
!pip uninstall -y codebleu tree-sitter tree-sitter-java tree-sitter-python -q
!pip install -q codebleu
!pip install -q "tree-sitter-java==0.21.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 3.4 MB/s eta 0:00:00


In [19]:
from codebleu import calc_codebleu

result = calc_codebleu(
    references=[[r] for r in references],   # list of lists, same shape as sacrebleu refs
    predictions=predictions,
    lang="java",
    weights=(0.25, 0.25, 0.25, 0.25),        # equal weight to ngram/weighted-ngram/AST/dataflow
    tokenizer=None
)

print('\n' + '─' * 50)
print('CodeBLEU Breakdown')
print('─' * 50)
print(f'  CodeBLEU          : {result["codebleu"]*100:.4f}')
print(f'  N-gram match      : {result["ngram_match_score"]*100:.4f}')
print(f'  Weighted n-gram    : {result["weighted_ngram_match_score"]*100:.4f}')
print(f'  Syntax (AST) match : {result["syntax_match_score"]*100:.4f}')
print(f'  Dataflow match     : {result["dataflow_match_score"]*100:.4f}')
print('─' * 50)


──────────────────────────────────────────────────
CodeBLEU Breakdown
──────────────────────────────────────────────────
  CodeBLEU          : 72.9968
  N-gram match      : 76.0529
  Weighted n-gram    : 76.3530
  Syntax (AST) match : 65.6601
  Dataflow match     : 73.9211
──────────────────────────────────────────────────


In [20]:
# import random

# random.seed(0)

# print(f"Total predictions: {len(predictions)} | Total references: {len(references)}\n")

# # 1. Print first 5 pairs, unfiltered
# print("=" * 70)
# print("FIRST 5 RAW PAIRS")
# print("=" * 70)
# for i in range(5):
#     print(f"\n--- Example {i} ---")
#     print(f"PRED [{len(predictions[i])} chars]: {predictions[i]!r}")
#     print(f"REF  [{len(references[i])} chars]: {references[i]!r}")

# # 2. Print 5 random pairs
# print("\n" + "=" * 70)
# print("5 RANDOM PAIRS")
# print("=" * 70)
# for i in random.sample(range(len(predictions)), 5):
#     print(f"\n--- Example {i} ---")
#     print(f"PRED [{len(predictions[i])} chars]: {predictions[i]!r}")
#     print(f"REF  [{len(references[i])} chars]: {references[i]!r}")

# # 3. Flag suspicious cases: prediction much longer/shorter than reference
# print("\n" + "=" * 70)
# print("SUSPICIOUS LENGTH MISMATCHES (pred >> ref or pred << ref)")
# print("=" * 70)
# flagged = []
# for i, (p, r) in enumerate(zip(predictions, references)):
#     if len(r) > 0 and (len(p) > 2 * len(r) or len(p) < 0.3 * len(r)):
#         flagged.append(i)

# print(f"Found {len(flagged)} flagged examples out of {len(predictions)}\n")
# for i in flagged[:5]:
#     print(f"\n--- Flagged Example {i} ---")
#     print(f"PRED [{len(predictions[i])} chars]: {predictions[i]!r}")
#     print(f"REF  [{len(references[i])} chars]: {references[i]!r}")

# # 4. Check for leftover template/prompt artifacts
# print("\n" + "=" * 70)
# print("CHECKING FOR LEFTOVER TEMPLATE TEXT")
# print("=" * 70)
# artifact_markers = ['Fix the following', '### Buggy Code', '### Fixed Code']
# artifact_count = 0
# for i, p in enumerate(predictions):
#     for marker in artifact_markers:
#         if marker in p:
#             artifact_count += 1
#             if artifact_count <= 3:
#                 print(f"\n--- Example {i} contains leftover marker {marker!r} ---")
#                 print(f"PRED: {p!r}")
#             break
# print(f"\nTotal predictions with leftover template artifacts: {artifact_count} / {len(predictions)}")

# # 5. Check for empty or near-empty predictions
# empty_count = sum(1 for p in predictions if len(p.strip()) == 0)
# print(f"\nEmpty predictions: {empty_count} / {len(predictions)}")

# # 6. Check average lengths
# avg_pred_len = sum(len(p) for p in predictions) / len(predictions)
# avg_ref_len  = sum(len(r) for r in references) / len(references)
# print(f"\nAverage prediction length: {avg_pred_len:.1f} chars")
# print(f"Average reference length : {avg_ref_len:.1f} chars")